# Fetch legrandaction.com HTML

This notebook provides two helper functions:

- `fetch_static_html(url)`: fetches the server-rendered HTML using `requests`.
- `get_rendered_html(url)`: fetches the JavaScript-rendered HTML using Playwright (runs a headless browser).

Use the code cell below to call either function and save or inspect the returned HTML.

In [ ]:
# Fetch helpers for https://www.legrandaction.com/
from __future__ import annotations

import asyncio
import re
from typing import Optional, Iterable, List, Dict

import requests

# Allow nested event loop in Jupyter so asyncio.run can be used safely.
# This uses nest_asyncio (already in requirements.txt).
try:
    import nest_asyncio

    # Apply nest_asyncio only when running under an existing event loop (e.g. Jupyter).
    try:
        nest_asyncio.apply()
    except Exception:
        # If applying fails, we'll still try to use the async functions directly.
        pass
except Exception:
    # nest_asyncio may not be installed in the environment; it's optional but
    # recommended if you want to run the rendered fetch from a running loop.
    pass


# Notebook-safe runner for async coroutines.
# Use this from sync wrapper functions instead of asyncio.run() to avoid
# "asyncio.run() cannot be called from a running event loop" errors in Jupyter.
def _run_sync(coro):
    """Run an async coroutine from sync code in a notebook-friendly way.

    If there's no running loop, use asyncio.run(). If a loop is running
    (typical in Jupyter), attempt to use the existing loop's run_until_complete.
    This relies on nest_asyncio.apply() having been called above.
    """
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        return asyncio.run(coro)

    if loop.is_running():
        try:
            return loop.run_until_complete(coro)
        except Exception:
            # Fallback: schedule a task and wait for it (requires nest_asyncio).
            task = asyncio.ensure_future(coro)
            return loop.run_until_complete(task)
    else:
        return asyncio.run(coro)


def fetch_static_html(url: str, timeout: int = 15) -> str:
    """Fetch HTML using a simple HTTP GET request.

    Returns the raw HTML as returned by the server.
    """
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        )
    }
    resp = requests.get(url, headers=headers, timeout=timeout)
    resp.raise_for_status()
    return resp.text


async def _fetch_rendered_html_async(url: str, timeout: int = 30, wait_until: Optional[str] = "networkidle") -> str:
    """Use Playwright to fetch the fully rendered HTML.

    If you plan to use this in the notebook, install Playwright and browsers first:
        python -m pip install playwright
        python -m playwright install
    """
    try:
        from playwright.async_api import async_playwright
    except Exception as e:
        raise RuntimeError("Playwright is not installed or failed to import") from e

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        try:
            context = await browser.new_context()
            page = await context.new_page()
            await page.goto(url, timeout=timeout * 1000, wait_until=wait_until)
            try:
                await page.wait_for_load_state(wait_until, timeout=5_000)
            except Exception:
                # Non-fatal; continue to grab content
                pass
            content = await page.content()
            return content
        finally:
            await browser.close()


def get_rendered_html(url: str, timeout: int = 30, wait_until: Optional[str] = "networkidle") -> str:
    """Sync wrapper around the Playwright async renderer.

    This wrapper uses a notebook-safe runner so it works inside Jupyter.
    If you're running in a pure Python process, behavior is unchanged.
    """
    return _run_sync(_fetch_rendered_html_async(url, timeout=timeout, wait_until=wait_until))


async def _fetch_rendered_all_tabs_async(url: str, timeout: int = 30, wait_until: Optional[str] = "networkidle", click_delay: float = 0.5) -> str:
    """Render the page and click all detected tab elements to collect each tab's content.

    Strategy:
    - Navigate to the page.
    - Find candidate tab controls (common selectors + fallback to visible anchors/buttons).
    - Click each candidate, wait briefly, then capture the current `document.body.innerHTML`.
    - Build and return a combined HTML document that contains each captured body in its own <section>.

    Returns:
        A single HTML string containing the original head and a body made of multiple
        <section data-tab-name="..."> blocks, one per detected tab.
    """
    try:
        from playwright.async_api import async_playwright
    except Exception as e:
        raise RuntimeError("Playwright is not installed or failed to import") from e

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        try:
            context = await browser.new_context()
            page = await context.new_page()
            await page.goto(url, timeout=timeout * 1000, wait_until=wait_until)
            # let initial scripts run
            await page.wait_for_timeout(500)

            # Candidate selectors that commonly represent tabs
            selectors = [
                '[role="tab"]',
                '[data-toggle="tab"]',
                '[data-role="tab"]',
                'ul.tabs li a',
                '.nav-tabs a',
                'nav a',
                'button[class*="tab"]',
                'a[class*="tab"]',
                '.tabs a',
                '.tab a',
            ]
            query = ','.join(selectors)
            elements = await page.query_selector_all(query)

            candidates: list[tuple] = []
            seen: set[str] = set()

            # collect with text and ensure visible
            for el in elements:
                try:
                    text = (await el.inner_text()) or ''
                    text = text.strip()
                    if not text:
                        continue
                    if text in seen:
                        continue
                    bbox = await el.bounding_box()
                    if not bbox:
                        continue
                    seen.add(text)
                    candidates.append((el, text))
                except Exception:
                    continue

            # Fallback: scan more broadly for visible anchors/buttons with short text
            if not candidates:
                elements = await page.query_selector_all('a, button, li')
                for el in elements:
                    try:
                        text = (await el.inner_text()) or ''
                        text = text.strip()
                        if not text or len(text) > 40:
                            continue
                        if text in seen:
                            continue
                        bbox = await el.bounding_box()
                        if not bbox:
                            continue
                        seen.add(text)
                        candidates.append((el, text))
                        if len(candidates) >= 12:
                            break
                    except Exception:
                        continue

            tab_captures: list[tuple[str, str]] = []
            # Click each candidate and capture body HTML
            for el, name in candidates:
                try:
                    await el.click()
                    await page.wait_for_timeout(int(click_delay * 1000))
                    body_html = await page.evaluate('document.body.innerHTML')
                    tab_captures.append((name, body_html))
                except Exception:
                    # ignore failures for individual tabs
                    continue

            # If we didn't detect tabs or captures, return the normal rendered content
            if not tab_captures:
                return await page.content()

            head_html = await page.evaluate('document.head.outerHTML')
            sections: list[str] = []
            for name, body in tab_captures:
                safe_name = name.replace('"', '').replace("'", '')
                sections.append(f'<section data-tab-name="{safe_name}"><h1>{safe_name}</h1>{body}</section>')

            combined_body = '\n'.join(sections)
            final_html = f"<!doctype html><html>{head_html}<body>{combined_body}</body></html>"
            return final_html
        finally:
            await browser.close()


def get_rendered_all_tabs_html(url: str, timeout: int = 30, wait_until: Optional[str] = "networkidle", click_delay: float = 0.5) -> str:
    """Sync wrapper to fetch and combine all tab content into one HTML string."""
    return _run_sync(_fetch_rendered_all_tabs_async(url, timeout=timeout, wait_until=wait_until, click_delay=click_delay))


# -- Google showtimes scraper --
async def _fetch_google_showtimes_async(cinema: str, location: Optional[str] = None, timeout: int = 30, wait_until: Optional[str] = "networkidle") -> dict:
    """Use Playwright to run a Google search for 'showtimes <cinema> <location>' and scrape times.

    Returns a mapping {movie_title: [time, ...], ...} discovered on the page.
    This is heuristic-based and may need tweaks per site/locale.
    """
    try:
        import urllib.parse
        from playwright.async_api import async_playwright
    except Exception as e:
        raise RuntimeError("Playwright is not installed or failed to import") from e

    query = f"showtimes {cinema} {location or ''}".strip()
    url = "https://www.google.com/search?q=" + urllib.parse.quote(query) + "&hl=en"

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        try:
            context = await browser.new_context()
            page = await context.new_page()
            await page.goto(url, timeout=timeout * 1000, wait_until=wait_until)
            await page.wait_for_timeout(800)

            # JS evaluation: find pieces of text that look like times and try to associate them with nearby headings
            js = r"""
(function(){
  const timeRegex = /\b\d{1,2}[:h]\d{2}\b/g;
  const nodes = Array.from(document.querySelectorAll('body *'));
  const results = [];

  function findTitle(node){
    // look for heading descendants
    const heading = node.querySelector('h1,h2,h3,h4,h5,strong,div[role="heading"]');
    if(heading && heading.innerText && heading.innerText.trim()) return heading.innerText.trim();
    // check previous siblings
    let p = node.previousElementSibling;
    while(p){
      const t = (p.innerText||'').trim();
      if(t && t.length>1 && t.length<100 && !timeRegex.test(t)) return t;
      p = p.previousElementSibling;
    }
    // climb up and search
    let parent = node.parentElement;
    for(let i=0;i<6 && parent;i++){
      const h = parent.querySelector('h1,h2,h3,h4,h5,strong,div[role="heading"]');
      if(h && h.innerText && h.innerText.trim()) return h.innerText.trim();
      parent = parent.parentElement;
    }
    return null;
  }

  const seenTitles = new Map();
  for(const node of nodes){
    const text = node.innerText||'';
    const times = text.match(timeRegex);
    if(times){
      const title = findTitle(node) || '';
      if(!title) continue;
      if(!seenTitles.has(title)) seenTitles.set(title, new Set());
      times.forEach(t=>seenTitles.get(title).add(t));
    }
  }

  const out = {};
  for(const [title,setTimes] of seenTitles){
    out[title] = Array.from(setTimes);
  }
  return out;
})();
"""

            captures = await page.evaluate(js)
            return captures
        finally:
            await browser.close()


def get_google_showtimes(cinema: str, location: Optional[str] = None, timeout: int = 30, wait_until: Optional[str] = "networkidle") -> dict:
    """Sync wrapper for Google showtimes scraper."""
    return _run_sync(_fetch_google_showtimes_async(cinema, location=location, timeout=timeout, wait_until=wait_until))


# Time normalization utilities
_time_h_re = re.compile(r"^(\d{1,2})h(\d{2})$")
_time_colon_re = re.compile(r"^(\d{1,2}):(\d{2})$")
_time_space_ampm_re = re.compile(r"^(\d{1,2}):(\d{2})\s*([ap]m)?$", re.I)


def normalize_time_string(s: str) -> Optional[str]:
    """Normalize a single time-like string to HH:MM (24h-ish) if possible.

    Handles examples like: '14h30', '14:30', '2:30', '2h30', '02:30', '14h'.
    If parsing is ambiguous (e.g., '2:30' with no AM/PM), the function will return it as-is
    in 'H:MM' form (not coercing to AM/PM).
    Returns None if no sensible time found.
    """
    if not s or not isinstance(s, str):
        return None
    s = s.strip()
    m = _time_h_re.match(s)
    if m:
        h = int(m.group(1))
        m2 = int(m.group(2))
        return f"{h:02d}:{m2:02d}"
    m = _time_colon_re.match(s)
    if m:
        h = int(m.group(1))
        m2 = int(m.group(2))
        return f"{h:02d}:{m2:02d}"
    # fallback: find a substring that looks like time
    m = re.search(r"(\d{1,2}[:h]\d{2})", s)
    if m:
        return normalize_time_string(m.group(1))
    return None


def time_to_minutes(t: str) -> Optional[int]:
    """Convert a normalized HH:MM string to minutes since midnight.
    Returns None on failure."""
    if not t:
        return None
    parts = t.split(":")
    if len(parts) != 2:
        return None
    try:
        h = int(parts[0])
        m = int(parts[1])
        return h * 60 + m
    except Exception:
        return None


def normalize_and_sort_times(times: Iterable[str]) -> List[str]:
    """Normalize and return a sorted list of time strings (HH:MM) from the input iterable."""
    out: List[tuple[int, str]] = []
    for s in times:
        norm = normalize_time_string(s)
        if not norm:
            continue
        mins = time_to_minutes(norm)
        if mins is None:
            continue
        out.append((mins, norm))
    out.sort()
    # dedupe while preserving order
    seen = set()
    res: List[str] = []
    for _, t in out:
        if t in seen:
            continue
        seen.add(t)
        res.append(t)
    return res


# DOM inspection helper to recommend selectors for Google showtimes
async def _inspect_google_dom_async(cinema: str, location: Optional[str] = None, timeout: int = 30) -> dict:
    """Navigate to Google showtimes search and return candidate selector paths that contain time-like strings.

    Returns a dict with samples for each suggested selector (truncated text).
    This is intended to be run manually in the notebook environment to help tune the scraper.
    """
    try:
        import urllib.parse
        from playwright.async_api import async_playwright
    except Exception as e:
        raise RuntimeError("Playwright is not installed or failed to import") from e

    query = f"showtimes {cinema} {location or ''}".strip()
    url = "https://www.google.com/search?q=" + urllib.parse.quote(query) + "&hl=en"

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        try:
            context = await browser.new_context()
            page = await context.new_page()
            await page.goto(url, timeout=timeout * 1000)
            await page.wait_for_timeout(1200)

            js = r"""
(function(){
  const timeRegex = /\b\d{1,2}[:h]\d{2}\b/g;
  const candidates = {};
  const nodes = Array.from(document.querySelectorAll('body *'));
  function pathFor(el){
    const parts = [];
    let cur = el;
    for(let i=0;i<6 && cur;i++){
      let part = cur.tagName.toLowerCase();
      if(cur.className && typeof cur.className==='string'){
        const cls = cur.className.trim().split(/\s+/).slice(0,2).join('.');
        if(cls) part += '.'+cls;
      }
      parts.unshift(part);
      cur = cur.parentElement;
    }
    return parts.join(' > ');
  }
  for(const n of nodes){
    const txt = (n.innerText||'').trim();
    if(!txt) continue;
    if(timeRegex.test(txt)){
      const p = pathFor(n);
      if(!candidates[p]) candidates[p] = (txt.length>200?txt.slice(0,200)+"...":txt);
    }
  }
  return candidates;
})();
"""
            candidates = await page.evaluate(js)
            # also save a snapshot to disk for manual inspection (optional)
            try:
                html = await page.content()
                # write to file in repo root
                with open('google_showtimes_inspect.html', 'w', encoding='utf-8') as f:
                    f.write(html)
            except Exception:
                pass
            return candidates
        finally:
            await browser.close()


def inspect_google_dom(cinema: str, location: Optional[str] = None, timeout: int = 30) -> dict:
    """Sync wrapper for DOM inspection helper."""
    return _run_sync(_inspect_google_dom_async(cinema, location=location, timeout=timeout))


# End of helper cell


In [21]:
# Example usage (run in notebook)
# This cell runs a static fetch by default and saves the HTML to a file.
# To test the JS-rendered fetch, set `use_rendered = True` (requires Playwright + browsers).

url = "https://www.legrandaction.com/"
url = "https://dulaccinemas.com/cinema/reflet-medicis/2950"
url = "https://espacesaintmichel.com/"
url = "https://www.cine-epeedebois.fr/"
url = "https://www.cinema-lechampo.com/films/prochainement.html#/" # problem
url = "https://www.lafilmotheque.fr/#front-next"
url ="https://www.lafilmotheque.fr/#"
url = "https://lafilmotheque-vad.cotecine.fr/reserver/"
# Christine - image. - https://pariscinemaclub.com/programmation-et-horaires/
url = "https://pariscinemaclub.com/ecoles-cinema-club/"
url = "http://studiogalande.fr/FR/43/horaires-cinema-studio-galande-beruchetparis.html"
out_static = "cinema_static.html"
out_rendered = "cinema_rendered.html"
out_all = "cinema_all_tabs.html"
print("Fetching static HTML...")
try:
    html_static = fetch_static_html(url)
    print(f"Static HTML fetched: {len(html_static)} characters")
    # Save to file for later processing or sending to OpenAI
    
    with open(out_static, "w", encoding="utf-8") as f:
        f.write(html_static)
    print(f"Saved static HTML to {out_static}")
except Exception as e:
    print("Static fetch failed:", e)

# Option 1: Fetch rendered HTML (single capture) - executes page JS and returns page.content()
use_rendered = False
if use_rendered:
    print("Fetching rendered HTML (this uses Playwright, may be slow)...")
    try:
        html_rendered = get_rendered_html(url)
        print(f"Rendered HTML fetched: {len(html_rendered)} characters")
        
        with open(out_rendered, "w", encoding="utf-8") as f:
            f.write(html_rendered)
        print(f"Saved rendered HTML to {out_rendered}")
    except Exception as e:
        print("Rendered fetch failed:", e)

# Option 2: Fetch rendered HTML and capture all tabs' content into one HTML
use_rendered_all_tabs = False
if use_rendered_all_tabs:
    print("Fetching rendered HTML with all tabs (Playwright, may be slower)...")
    try:
        html_all_tabs = get_rendered_all_tabs_html(url)
        print(f"All-tabs HTML fetched: {len(html_all_tabs)} characters")
        
        with open(out_all, "w", encoding="utf-8") as f:
            f.write(html_all_tabs)
        print(f"Saved combined tabs HTML to {out_all}")
    except Exception as e:
        print("Rendered all-tabs fetch failed:", e)

# Quick preview: print the first 500 characters of the static HTML
print('\n--- Static HTML preview (first 500 chars) ---')
print(html_static[:500])

# --- New example: get Google showtimes for a cinema (no API) ---
try:
    cinema_name = "Arlequin"
    location = "Paris"
    print(f"Fetching Google showtimes for: {cinema_name}, {location}")
    showtimes = get_google_showtimes(cinema_name, location)
    import json
    # Normalize and sort times per movie
    normalized = {title: normalize_and_sort_times(times) for title, times in showtimes.items()}
    print(json.dumps(normalized, indent=2, ensure_ascii=False))
except Exception as e:
    print("Google showtimes fetch failed:", e)

# --- Optional: run a DOM inspection to gather candidate selectors for tuning the scraper ---
# Uncomment and run the next block if you want to perform a live inspection and create
# `google_showtimes_inspect.html` alongside a returned dict of candidate selector paths.
# Requires Playwright + browsers to be installed.
#
# try:
#     print('Inspecting Google DOM for selector candidates...')
#     candidates = inspect_google_dom('Arlequin', 'Paris')
#     import json
#     print('Found candidate selector paths (sample):')
#     print(json.dumps({k: candidates[k][:400] if isinstance(candidates[k], str) else candidates[k] for k in list(candidates)[:20]}, indent=2, ensure_ascii=False))
# except Exception as e:
#     print('DOM inspection failed:', e)


Fetching static HTML...
Static HTML fetched: 103103 characters
Saved static HTML to cinema_static.html
Fetching rendered HTML with all tabs (Playwright, may be slower)...
All-tabs HTML fetched: 687676 characters
Saved combined tabs HTML to cinema_all_tabs.html

--- Static HTML preview (first 500 chars) ---
﻿<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" lang="FR">
<head>
<title>Les horaires du cinéma Studio-galande Béruchet à Paris</title>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<meta http-equiv="content-language" content="fr" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta name="keyword" content="programmation,horaires,films,cinéma,Stu
All-tabs HTML fetched: 687676 characters
Saved combined tabs HTML to cinema_all_tabs.html

--- Static HTML preview (first 500 chars) ---
﻿<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transiti

In [9]:
import os
import requests

# Read SerpAPI key from the environment to avoid hardcoding secrets in the notebook.
# Set it in your shell before launching Jupyter, e.g. (zsh):
#   export SERPAPI_KEY="<your-key-here>"
# Or use a .env loader if you prefer (not included here).

API_KEY = 'bd09d0757c9108469a51dc6c0102ab5c3560c16f5e746f1fed1fe31c38c5c46d'
if not API_KEY:
    print("SERPAPI_KEY environment variable is not set.\n"
          "Set it in your shell before launching Jupyter, for example:\n"
          "  export SERPAPI_KEY=\"your_real_key_here\"")
else:
    params = {
        "engine": "google_showtimes",
        "q": "Showtimes at L'Arlequin",
        "location": "Paris, France",
        "hl": "en",
        "api_key": API_KEY,
    }

    try:
        r = requests.get("https://serpapi.com/search.json", params=params, timeout=30)
        # Raise for status to catch HTTP errors below
        r.raise_for_status()
    except requests.HTTPError as http_err:
        # Provide helpful diagnostics for 401 Unauthorized or other status codes
        status = getattr(http_err.response, 'status_code', 'N/A') if hasattr(http_err, 'response') else 'N/A'
        body = http_err.response.text if (hasattr(http_err, 'response') and http_err.response is not None) else 'N/A'
        print(f"SerpAPI request failed: {http_err}\nStatus code: {status}\nResponse body (truncated): {body[:500]!r}")
        if status == 401:
            print("401 Unauthorized — your API key may be invalid or revoked.\n"
                  "Check that SERPAPI_KEY is correct and has remaining quota.")
    except Exception as e:
        print("Failed to contact SerpAPI:", e)
    else:
        try:
            data = r.json()
            # Depending on the response shape, you'll typically find theaters/movies/showtimes
            print('Response keys:', list(data.keys()))
        except Exception as e:
            print("Failed to decode SerpAPI response as JSON:", e)


SerpAPI request failed: 400 Client Error: Bad Request for url: https://serpapi.com/search.json?engine=google_showtimes&q=Showtimes+at+L%27Arlequin&location=Paris%2C+France&hl=en&api_key=bd09d0757c9108469a51dc6c0102ab5c3560c16f5e746f1fed1fe31c38c5c46d
Status code: 400
Response body (truncated): '{\n  "error": "Unsupported `google_showtimes` search engine."\n}'


In [ ]:
### Google showtimes scraper and utilities (separate cell)
# This cell provides a self-contained Playwright-based Google showtimes scraper
# and time-normalization utilities. Run this cell to (re)define the functions.

from __future__ import annotations
import asyncio
import re
from typing import Optional, Iterable, List, Dict, Any

# Time normalization utilities
_time_h_re = re.compile(r"^(\d{1,2})h(\d{2})$")
_time_colon_re = re.compile(r"^(\d{1,2}):(\d{2})$")
_time_space_ampm_re = re.compile(r"^(\d{1,2}):(\d{2})\s*([ap]m)?$", re.I)
TIME_JS_RE = r"\\b\\d{1,2}[:h]\\d{2}\\b"

def normalize_time_string(s: str) -> Optional[str]:
    """Normalize a single time-like string to HH:MM (24h-ish) if possible."""
    if not s or not isinstance(s, str):
        return None
    s = s.strip()
    m = _time_h_re.match(s)
    if m:
        h = int(m.group(1))
        m2 = int(m.group(2))
        return f"{h:02d}:{m2:02d}"
    m = _time_colon_re.match(s)
    if m:
        h = int(m.group(1))
        m2 = int(m.group(2))
        return f"{h:02d}:{m2:02d}"
    m = re.search(r"(\\d{1,2}[:h]\\d{2})", s)
    if m:
        return normalize_time_string(m.group(1))
    return None

def time_to_minutes(t: str) -> Optional[int]:
    if not t:
        return None
    parts = t.split(":")
    if len(parts) != 2:
        return None
    try:
        h = int(parts[0])
        m = int(parts[1])
        return h * 60 + m
    except Exception:
        return None

def normalize_and_sort_times(times: Iterable[str]) -> List[str]:
    out: List[tuple[int, str]] = []
    for s in times:
        norm = normalize_time_string(s)
        if not norm:
            continue
        mins = time_to_minutes(norm)
        if mins is None:
            continue
        out.append((mins, norm))
    out.sort()
    seen = set()
    res: List[str] = []
    for _, t in out:
        if t in seen:
            continue
        seen.add(t)
        res.append(t)
    return res

async def _fetch_google_showtimes_async(cinema: str, location: Optional[str] = None, timeout: int = 30, wait_until: Optional[str] = "networkidle") -> Dict[str, Dict[str, List[str]]]:
    """Use Playwright to run a Google search for 'showtimes <cinema> <location>' and scrape times.

    Returns a mapping {day_label: {movie_title: [time, ...], ...}, ...}.
    Strategy:
      - Load the Google search showtimes page.
      - Locate an element that contains time-like strings to find the showtimes area.
      - Find clickable candidate elements (tabs) near that area and click each, capturing times after each click.
      - Return per-tab captures. If no tab candidates are found, return a single 'default' key with the captured times.
    """
    try:
        import urllib.parse
        from playwright.async_api import async_playwright
    except Exception as e:
        raise RuntimeError("Playwright is not installed or failed to import") from e

    query = f"showtimes {cinema} {location or ''}".strip()
    url = "https://www.google.com/search?q=" + urllib.parse.quote(query) + "&hl=en"

    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        try:
            context = await browser.new_context()
            page = await context.new_page()
            await page.goto(url, timeout=timeout * 1000, wait_until=wait_until)
            await page.wait_for_timeout(900)

            # Helper JS to extract movie->times mapping from the currently visible DOM
            # Build the JS string without using Python f-strings to avoid brace escaping issues
            extract_js = (
                "(function(){\n"
                "  const timeRegex = /" + TIME_JS_RE + "/g;\n"
                "  const nodes = Array.from(document.querySelectorAll('body *'));\n"
                "  const seenTitles = new Map();\n"
                "  function findTitle(node){\n"
                "    const heading = node.querySelector('h1,h2,h3,h4,h5,strong,div[role=\"heading\"]');\n"
                "    if(heading && heading.innerText && heading.innerText.trim()) return heading.innerText.trim();\n"
                "    let p = node.previousElementSibling;\n"
                "    while(p){\n"
                "      const t = (p.innerText||'').trim();\n"
                "      if(t && t.length>1 && t.length<100 && !timeRegex.test(t)) return t;\n"
                "      p = p.previousElementSibling;\n"
                "    }\n"
                "    let parent = node.parentElement;\n"
                "    for(let i=0;i<6 && parent;i++){\n"
                "      const h = parent.querySelector('h1,h2,h3,h4,h5,strong,div[role=\"heading\"]');\n"
                "      if(h && h.innerText && h.innerText.trim()) return h.innerText.trim();\n"
                "      parent = parent.parentElement;\n"
                "    }\n"
                "    return null;\n"
                "  }\n"
                "  for(const node of nodes){\n"
                "    const text = node.innerText||'';\n"
                "    const times = text.match(timeRegex);\n"
                "    if(times){\n"
                "      const title = findTitle(node) || '';\n"
                "      if(!title) continue;\n"
                "      if(!seenTitles.has(title)) seenTitles.set(title, new Set());\n"
                "      times.forEach(t=>seenTitles.get(title).add(t));\n"
                "    }\n"
                "  }\n"
                "  const out = {};\n"
                "  for(const [title,setTimes] of seenTitles){\n"
                "    out[title] = Array.from(setTimes);\n"
                "  }\n"
                "  return out;\n"
                "})();"
            )

            # Capture the 'default' set (without clicking any tabs)
            try:
                default_capture = await page.evaluate(extract_js)
            except Exception:
                default_capture = {}

            # Try to locate an element that contains times to identify the showtimes area
            first_time_box = None
            try:
                # scan a limited set of elements to find a time-containing node and its bbox
                all_nodes = await page.query_selector_all('body *')
                for n in all_nodes:
                    try:
                        text = (await n.inner_text()) or ''
                        if re.search(r"\\b\\d{1,2}[:h]\\d{2}\\b", text):
                            bbox = await n.bounding_box()
                            if bbox:
                                first_time_box = bbox
                                break
                    except Exception:
                        continue
            except Exception:
                first_time_box = None

            tab_results: Dict[str, Dict[str, List[str]]] = {}
            # Always include the default capture under a descriptive key
            tab_results['default'] = default_capture if isinstance(default_capture, dict) else {}

            # If we didn't find a time area, return the default capture
            if not first_time_box:
                return tab_results

            # Find candidate clickable elements near the time area (anchors/buttons)
            try:
                candidates = []
                candidates_el = await page.query_selector_all('a,button')
                seen_texts = set()
                center_y = first_time_box['y'] + first_time_box['height']/2
                for el in candidates_el:
                    try:
                        bbox = await el.bounding_box()
                        if not bbox:
                            continue
                        # only consider elements roughly horizontally overlapping and vertically near the showtimes area
                        el_center_y = bbox['y'] + bbox['height']/2
                        if abs(el_center_y - center_y) > 220:
                            continue
                        text = (await el.inner_text()) or ''
                        text = text.strip()
                        if not text or len(text) > 40:
                            continue
                        if text in seen_texts:
                            continue
                        seen_texts.add(text)
                        candidates.append((el, text))
                        if len(candidates) >= 12:
                            break
                    except Exception:
                        continue
            except Exception:
                candidates = []

            # Click each candidate and capture the resulting times. Use a small delay after each click.
            for el, name in candidates:
                try:
                    await el.click()
                    await page.wait_for_timeout(650)
                    try:
                        cap = await page.evaluate(extract_js)
                    except Exception:
                        cap = {}
                    # Only store captures that contain something different from the default (or non-empty)
                    if cap and cap != tab_results.get('default'):
                        tab_results[name] = cap
                except Exception:
                    continue

            # If we didn't find any distinct tab captures, return just the default mapping
            if len(tab_results) <= 1:
                return tab_results

            return tab_results
        finally:
            await browser.close()


def get_google_showtimes(cinema: str, location: Optional[str] = None, timeout: int = 30, wait_until: Optional[str] = "networkidle") -> Dict[str, Dict[str, List[str]]]:
    """Sync wrapper for Google showtimes scraper returning per-day captures."""
    return _run_sync(_fetch_google_showtimes_async(cinema, location=location, timeout=timeout, wait_until=wait_until))

# End of helper cell


In [12]:
### Example: run Google showtimes extraction
# Run this cell after the Google scraper cell above. It will attempt to fetch showtimes
# for the given cinema (uses Playwright).

cinema_name = "Arlequin"
location = "Paris"

print("Fetching showtimes for:", cinema_name, location)
try:
    results = get_google_showtimes(cinema_name, location)
    from pprint import pprint
    pprint(results)
    # Normalize and sort times per title
    normalized = {title: normalize_and_sort_times(times) for title, times in results.items()}
    print("\nNormalized & sorted:")
    pprint(normalized)
except Exception as e:
    print("Error running Playwright scraper in this environment:", e)
    print("If you're running this in Jupyter locally make sure Playwright and browsers are installed:")
    print("  python -m pip install playwright\n  python -m playwright install")
    print("Or run the inspection helper cell to produce selector candidates:")
    print("  inspect_google_dom(\"Arlequin\", \"Paris\")")


Fetching showtimes for: Arlequin Paris
Error running Playwright scraper in this environment: asyncio.run() cannot be called from a running event loop
If you're running this in Jupyter locally make sure Playwright and browsers are installed:
  python -m pip install playwright
  python -m playwright install
Or run the inspection helper cell to produce selector candidates:
  inspect_google_dom("Arlequin", "Paris")


/var/folders/90/w4_gkkk904b34gjm9tlbc67r0000gn/T/ipykernel_14018/2334378273.py:22: RuntimeWarning: coroutine '_fetch_google_showtimes_async' was never awaited
  print("  inspect_google_dom(\"Arlequin\", \"Paris\")")


In [13]:
# SerpAPI-based showtimes fetch and diagnostic cell
# Use this cell to call SerpAPI (engine='google') and extract time-like strings from the JSON response.

import os
import requests
import re
import json
from typing import Tuple, Dict, Any, Optional

TIME_RE = re.compile(r"\b\d{1,2}[:h]\d{2}\s*(?:[ap]m)?\b", re.I)


def get_showtimes_serpapi(cinema: str, location: Optional[str] = None, api_key: Optional[str] = None, timeout: int = 30) -> Tuple[Dict[str, Any], Dict[str, list]]:
    """Query SerpAPI using engine='google' and return (raw_json, matches).

    - Saves `serpapi_showtimes.json` in the repo for inspection.
    - Returns a mapping of JSON-path -> list of time-like strings found there.
    """
    if api_key is None:
        api_key = os.getenv('SERP_API')
    if not api_key:
        raise RuntimeError('SERP_API environment variable is not set')

    q = f"showtimes {cinema} {location or ''}".strip()
    params = {
        'engine': 'google',
        'q': q,
        'hl': 'en',
        'api_key': api_key,
    }
    if location:
        params['location'] = location

    try:
        r = requests.get('https://serpapi.com/search.json', params=params, timeout=timeout)
        r.raise_for_status()
    except requests.HTTPError as http_err:
        # print helpful diagnostics including raw response body when available
        status = getattr(http_err.response, 'status_code', 'N/A') if hasattr(http_err, 'response') else 'N/A'
        body = http_err.response.text if (hasattr(http_err, 'response') and http_err.response is not None) else ''
        print(f"SerpAPI request failed: {http_err}\nStatus code: {status}\nResponse body (truncated): {body[:1000]!r}")
        raise
    except Exception as e:
        print('Failed to contact SerpAPI:', e)
        raise

    data = r.json()
    # Save raw response for inspection
    try:
        with open('serpapi_showtimes.json', 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print('Saved SerpAPI raw response to serpapi_showtimes.json')
    except Exception as e:
        print('Failed to write serpapi_showtimes.json:', e)

    matches: dict[str, list] = {}

    def traverse(obj, path='root'):
        if isinstance(obj, str):
            found = TIME_RE.findall(obj)
            if found:
                matches[path] = found
        elif isinstance(obj, dict):
            for k, v in obj.items():
                traverse(v, f"{path}/{k}")
        elif isinstance(obj, list):
            for i, v in enumerate(obj):
                traverse(v, f"{path}[{i}]")

    traverse(data, 'root')

    print('Top-level keys in SerpAPI response:', list(data.keys()))
    print(f'Found {len(matches)} JSON paths containing time-like strings (sample up to 30):')
    import pprint
    pprint.pprint({k: matches[k] for k in list(matches)[:30]})

    return data, matches


# Example usage (adjust cinema/location as needed)
try:
    cinema = "L'Arlequin"
    location = "Paris, France"
    data, matches = get_showtimes_serpapi(cinema, location)
except Exception as e:
    print('SerpAPI fetch error:', e)


Saved SerpAPI raw response to serpapi_showtimes.json
Top-level keys in SerpAPI response: ['search_metadata', 'search_parameters', 'search_information', 'showtimes', 'knowledge_graph', 'related_questions', 'ai_overview', 'organic_results', 'menu_highlights', 'related_searches', 'pagination', 'serpapi_pagination']
Found 41 JSON paths containing time-like strings (sample up to 30):
{'root/search_metadata/created_at': ['14:51'],
 'root/search_metadata/processed_at': ['14:51'],
 'root/showtimes[0]/movies[0]/showing[0]/time[0]': ['5:10pm'],
 'root/showtimes[0]/movies[0]/showing[0]/time[1]': ['7:15pm'],
 'root/showtimes[0]/movies[0]/showing[0]/time[2]': ['9:20pm'],
 'root/showtimes[0]/movies[1]/showing[0]/time[0]': ['4:20pm'],
 'root/showtimes[0]/movies[2]/showing[0]/time[0]': ['5:00pm'],
 'root/showtimes[0]/movies[2]/showing[0]/time[1]': ['8:15pm'],
 'root/showtimes[0]/movies[3]/showing[0]/time[0]': ['6:30pm'],
 'root/showtimes[0]/movies[4]/showing[0]/time[0]': ['8:50pm'],
 'root/showtimes[1

In [7]:
data

{'search_metadata': {'id': '693ecedaeeabd37af4e62e5e',
  'status': 'Success',
  'json_endpoint': 'https://serpapi.com/searches/934af70257811238/693ecedaeeabd37af4e62e5e.json',
  'pixel_position_endpoint': 'https://serpapi.com/searches/934af70257811238/693ecedaeeabd37af4e62e5e.json_with_pixel_position',
  'created_at': '2025-12-14 14:51:06 UTC',
  'processed_at': '2025-12-14 14:51:06 UTC',
  'google_url': 'https://www.google.com/search?q=showtimes+L%27Arlequin+Paris%2C+France&oq=showtimes+L%27Arlequin+Paris%2C+France&uule=w+CAIQICIgUGFyaXMsUGFyaXMsSWxlLWRlLUZyYW5jZSxGcmFuY2U&hl=en&sourceid=chrome&ie=UTF-8',
  'raw_html_file': 'https://serpapi.com/searches/934af70257811238/693ecedaeeabd37af4e62e5e.html',
  'total_time_taken': 1.32},
 'search_parameters': {'engine': 'google',
  'q': "showtimes L'Arlequin Paris, France",
  'location_requested': 'Paris, France',
  'location_used': 'Paris,Paris,Ile-de-France,France',
  'google_domain': 'google.com',
  'hl': 'en',
  'device': 'desktop'},
 'se

In [14]:
# Build a clean HTML table from SerpAPI / scraped results
# Produces `showtimes_table.html` with columns:
# Title | Director | Year | Country | Cinema | Days & Times

import os
import re
import json
import html
from typing import Any, Dict, List, Optional

TIME_RE = re.compile(r"\b\d{1,2}[:h]\d{2}\s*(?:[ap]m)?\b", re.I)
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")


def load_serpapi_data() -> Dict[str, Any]:
    # Prefer in-memory `data` if present (from previous cell), otherwise read saved file
    try:
        d = globals().get('data')
        if isinstance(d, dict):
            print('Using in-memory `data` object')
            return d
    except Exception:
        pass
    fname = 'serpapi_showtimes.json'
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            return json.load(f)
    raise RuntimeError('No SerpAPI data found in-memory or serpapi_showtimes.json')


def collect_text(obj: Any) -> str:
    # Recursively collect visible text from JSON node
    if obj is None:
        return ''
    if isinstance(obj, str):
        return obj
    if isinstance(obj, (int, float)):
        return str(obj)
    s = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            # skip large blobs like images
            if k.lower() in ('thumbnail', 'image', 'logo', 'url'):
                continue
            s.append(collect_text(v))
    elif isinstance(obj, list):
        for v in obj:
            s.append(collect_text(v))
    return '\n'.join([x for x in s if x])


def try_extract_metadata(text: str) -> Dict[str, Optional[str]]:
    # Heuristics to pick director, year, country from free text
    director = None
    year = None
    country = None
    # year
    m = YEAR_RE.search(text)
    if m:
        year = m.group(0)
    # director - look for 'Directed by X' or 'Dir. X' or 'réalisé par X'
    dm = re.search(r"(?:Directed by|Dir\.|Dir|réalisé par|réal\.|Réalisé par)\s+([^\n,\(]+)", text, re.I)
    if dm:
        director = dm.group(1).strip()
    # country - look for common country words or "France", "USA", etc.
    cm = re.search(r"\b(France|USA|United States|United Kingdom|UK|Germany|Italy|Spain|Canada|Belgium)\b", text, re.I)
    if cm:
        country = cm.group(1)
    return {'director': director, 'year': year, 'country': country}


def extract_candidates_from_json(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    candidates: List[Dict[str, Any]] = []

    def visit(node: Any, path: str = ''):
        # If node is dict and contains a title/name, consider it
        if isinstance(node, dict):
            title = None
            for key in ('title', 'name', 'movie_title'):
                if key in node and isinstance(node[key], str) and node[key].strip():
                    title = node[key].strip()
                    break
            # also consider nodes where a name appears nested in strings
            if title:
                text = collect_text(node)
                times = TIME_RE.findall(text)
                meta = try_extract_metadata(text)
                cinema = None
                # try to guess cinema from nearby fields
                for ck in ('cinema', 'theater', 'venue', 'place'):
                    if ck in node and isinstance(node[ck], str):
                        cinema = node[ck]
                        break
                candidates.append({
                    'title': title,
                    'director': meta.get('director'),
                    'year': meta.get('year'),
                    'country': meta.get('country'),
                    'cinema': cinema,
                    'times': sorted(set(TIME_RE.findall(text)))
                })
            # continue traversal
            for k, v in node.items():
                visit(v, f"{path}/{k}")
        elif isinstance(node, list):
            for i, v in enumerate(node):
                visit(v, f"{path}[{i}]")
        else:
            # primitives ignored at top-level
            pass

    visit(data, 'root')

    # If no explicit candidates found, try scanning top-level text fields for repeated titles
    if not candidates:
        text = collect_text(data)
        # naive split by lines containing a year or time - group by lines that look like titles
        lines = [l.strip() for l in text.splitlines() if l.strip()]
        # look for lines with colon-separated times and previous non-time line as title
        for i, line in enumerate(lines):
            if TIME_RE.search(line):
                # previous line may be title
                if i > 0 and not TIME_RE.search(lines[i-1]):
                    title = lines[i-1]
                    times = TIME_RE.findall(line)
                    meta = try_extract_metadata('\n'.join(lines[max(0, i-5):i+1]))
                    candidates.append({'title': title, 'director': meta.get('director'), 'year': meta.get('year'), 'country': meta.get('country'), 'cinema': None, 'times': times})

    # Deduplicate by title
    seen = {}
    out = []
    for c in candidates:
        key = (c.get('title') or '').lower()
        if not key:
            continue
        if key in seen:
            # merge times
            seen[key]['times'] = sorted(set(seen[key]['times']) | set(c.get('times', [])))
            for fld in ('director', 'year', 'country', 'cinema'):
                if not seen[key].get(fld) and c.get(fld):
                    seen[key][fld] = c.get(fld)
        else:
            seen[key] = c.copy()
    for v in seen.values():
        out.append(v)
    return out


def render_html_table(rows: List[Dict[str, Any]]) -> str:
    style = '''<style>
    table {border-collapse: collapse; width: 100%; font-family: Arial, sans-serif}
    th, td {border: 1px solid #ccc; padding: 6px; text-align: left}
    th {background: #f3f3f3}
    </style>'''
    parts = ["<html><head>", style, "</head><body>"]
    parts.append('<table>')
    parts.append('<thead><tr><th>Title</th><th>Director</th><th>Year</th><th>Country</th><th>Cinema</th><th>Days & Times</th></tr></thead>')
    parts.append('<tbody>')
    for r in rows:
        title = html.escape(r.get('title') or '')
        director = html.escape(r.get('director') or '')
        year = html.escape(r.get('year') or '')
        country = html.escape(r.get('country') or '')
        cinema = html.escape(r.get('cinema') or '')
        times = r.get('times') or []
        times_str = html.escape(', '.join(times))
        parts.append(f'<tr><td>{title}</td><td>{director}</td><td>{year}</td><td>{country}</td><td>{cinema}</td><td>{times_str}</td></tr>')
    parts.append('</tbody></table>')
    parts.append('</body></html>')
    return '\n'.join(parts)


# Main usage
try:
    data_obj = load_serpapi_data()
    candidates = extract_candidates_from_json(data_obj)
    if not candidates:
        print('No film candidates detected in SerpAPI response')
    else:
        print(f'Found {len(candidates)} film candidates')
    html_out = render_html_table(candidates)
    out_file = 'showtimes_table.html'
    with open(out_file, 'w', encoding='utf-8') as f:
        f.write(html_out)
    print(f'Wrote {out_file} (open in browser to view)')
except Exception as e:
    print('Failed to build showtimes table:', e)


Using in-memory `data` object
Found 32 film candidates
Wrote showtimes_table.html (open in browser to view)


In [ ]:
# Display the generated showtimes table inline in the notebook (quick preview)
from IPython.display import HTML, display
try:
    display(HTML(html_out))
except NameError:
    print('html_out is not defined. Run the previous cell that builds the showtimes table.')

In [16]:
# Build a film-centric table from per-day captures (Google Playwright output)
# Expects a variable `showtimes` in the notebook containing a mapping:
#   { day_label: { movie_title: [time, ...], ... }, ... }
# Example: showtimes = get_google_showtimes('Arlequin', 'Paris')
from IPython.display import HTML, display

showtimes = get_google_showtimes("L'Arlequin", "Paris")
try:
    day_captures = globals().get('showtimes')
    if not day_captures or not isinstance(day_captures, dict):
        raise NameError
except NameError:
    print("No per-day captures found in variable 'showtimes'. Run `showtimes = get_google_showtimes(cinema, location)` first and re-run this cell.")
else:
    # Merge into a movie-centric map
    movie_map = {}
    for day_label, movies in day_captures.items():
        if not isinstance(movies, dict):
            continue
        for title, times in movies.items():
            if not title or not isinstance(title, str):
                continue
            key = title.strip()
            if not key:
                continue
            entry = movie_map.setdefault(key, {
                'title': title.strip(),
                'director': None,
                'year': None,
                'country': None,
                'cinema': None,
                'schedule': {}
            })
            # Normalize times
            if not isinstance(times, (list, tuple)):
                continue
            norm = [t for t in (normalize_time_string(t) or '' for t in times) if t]
            if not norm:
                continue
            # dedupe and sort times for the day
            norm_sorted = normalize_and_sort_times(norm)
            if norm_sorted:
                entry['schedule'].setdefault(day_label, [])
                # merge preserving order
                existing = entry['schedule'][day_label]
                for t in norm_sorted:
                    if t not in existing:
                        existing.append(t)

    # Build rows for rendering
    rows = []
    for title, info in sorted(movie_map.items(), key=lambda x: x[0].lower()):
        schedule = info.get('schedule', {})
        # format day lines
        day_lines = []
        # keep day ordering as they appeared in the captures where possible
        for day in schedule:
            times_list = schedule.get(day) or []
            if not times_list:
                continue
            day_lines.append(f"{day}: {', '.join(times_list)}")
        rows.append({
            'title': info.get('title'),
            'director': info.get('director'),
            'year': info.get('year'),
            'country': info.get('country'),
            'cinema': info.get('cinema'),
            'times': day_lines
        })

    # Render an HTML table where Days & Times cell contains line breaks
    def render_film_centric_table(rows_list):
        style = '''<style>
        table {border-collapse: collapse; width: 100%; font-family: Arial, sans-serif}
        th, td {border: 1px solid #ccc; padding: 6px; text-align: left}
        th {background: #f3f3f3}
        </style>'''
        parts = ["<html><head>", style, "</head><body>"]
        parts.append('<table>')
        parts.append('<thead><tr><th>Title</th><th>Director</th><th>Year</th><th>Country</th><th>Cinema</th><th>Days & Times</th></tr></thead>')
        parts.append('<tbody>')
        import html as _html
        for r in rows_list:
            title = _html.escape(r.get('title') or '')
            director = _html.escape(r.get('director') or '')
            year = _html.escape(r.get('year') or '')
            country = _html.escape(r.get('country') or '')
            cinema = _html.escape(r.get('cinema') or '')
            times = r.get('times') or []
            # escape each line and join with <br>
            times_html = '<br>'.join([_html.escape(line) for line in times])
            parts.append(f"<tr><td>{title}</td><td>{director}</td><td>{year}</td><td>{country}</td><td>{cinema}</td><td>{times_html}</td></tr>")
        parts.append('</tbody></table>')
        parts.append('</body></html>')
        return '\n'.join(parts)

    html_film = render_film_centric_table(rows)
    out_file = 'showtimes_film_centric.html'
    try:
        with open(out_file, 'w', encoding='utf-8') as f:
            f.write(html_film)
        print(f'Wrote {out_file}')
    except Exception as e:
        print('Failed to write output file:', e)

    display(HTML(html_film))

RuntimeError: asyncio.run() cannot be called from a running event loop